<a href="https://colab.research.google.com/github/slomi23/ML_fx/blob/main/model_experiment_N_BEATS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone "https://github.com/slomi23/ML_fx.git"
!cd ML_fx/

fatal: destination path 'ML_fx' already exists and is not an empty directory.


## Fetch Data

In [ ]:
import pandas as pd
import numpy as np
import os
import zipfile
import io

PROCCESSED_DATA_DIR = "./ML_fx/data/processed/"
train=pd.read_csv(os.path.join(PROCCESSED_DATA_DIR, "train_prepared.csv"))
print(train.head())

   Store  Dept        Date  Weekly_Sales  IsHoliday  Temperature  Fuel_Price  \
0      1     1  2010-02-05      24924.50          0        42.31       2.572   
1      1     1  2010-02-12      46039.49          1        38.51       2.548   
2      1     1  2010-02-19      41595.55          0        39.93       2.514   
3      1     1  2010-02-26      19403.54          0        46.63       2.561   
4      1     1  2010-03-05      21827.90          0        46.50       2.625   

   MarkDown1  MarkDown2  MarkDown3  ...  Type    Size  sales_lag_52  Year  \
0    5347.45      192.0       24.6  ...    20  151315       7998.55  2010   
1    5347.45      192.0       24.6  ...    20  151315       7998.55  2010   
2    5347.45      192.0       24.6  ...    20  151315       7998.55  2010   
3    5347.45      192.0       24.6  ...    20  151315       7998.55  2010   
4    5347.45      192.0       24.6  ...    20  151315       7998.55  2010   

   month_sin     month_cos   dow_sin   dow_cos  week_sin

# W&B

In [ ]:
!pip install wandb -q
!pip install neuralforecast torch pytorch-lightning
!pip install pytorch-forecasting pandas numpy torch matplotlib


import wandb
import os

# Retrieve the secret from Kaggle Secrets

api_key = "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"
if api_key:
    wandb.login(key=api_key)
else:
    print("Warning: could not log in wandb ")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: slomi23 (slomi23-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


# Dropping the features and preparing for training and validation
N-BEATS is an univariate algorithm

In [ ]:
split_date = '2011-12-01'

# Create Train and Validation Sets
val_set = train[train['Date'] >= split_date]
train_set = train[train['Date'] < split_date]

y_train = train_set['Weekly_Sales']
X_train = train_set.drop(columns=['Weekly_Sales', 'Date'])
y_val = val_set['Weekly_Sales']
X_val = val_set.drop(columns=['Weekly_Sales', 'Date'])


print(f"Final Training Set Shape: {train_set.shape}")
print(f"Validation Set Shape: {val_set.shape}")
print(f"Validation Period: {val_set['Date'].min()} to {val_set['Date'].max()}")
print(X_train.head())

Final Training Set Shape: (279085, 24)
Validation Set Shape: (142485, 24)
Validation Period: 2011-12-02 to 2012-10-26
   Store  Dept  IsHoliday  Temperature  Fuel_Price  MarkDown1  MarkDown2  \
0      1     1          0        42.31       2.572    5347.45      192.0   
1      1     1          1        38.51       2.548    5347.45      192.0   
2      1     1          0        39.93       2.514    5347.45      192.0   
3      1     1          0        46.63       2.561    5347.45      192.0   
4      1     1          0        46.50       2.625    5347.45      192.0   

   MarkDown3  MarkDown4  MarkDown5  ...  Type    Size  sales_lag_52  Year  \
0       24.6    1481.31    3359.45  ...    20  151315       7998.55  2010   
1       24.6    1481.31    3359.45  ...    20  151315       7998.55  2010   
2       24.6    1481.31    3359.45  ...    20  151315       7998.55  2010   
3       24.6    1481.31    3359.45  ...    20  151315       7998.55  2010   
4       24.6    1481.31    3359.45  ... 

In [ ]:
from pytorch_forecasting.data import TimeSeriesDataSet
from pytorch_forecasting.models.nbeats import NBeats

df = train.copy()
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)

unique_dates = df['Date'].sort_values().unique()
date_to_time_idx = {date: idx for idx, date in enumerate(unique_dates)}
df['time_idx'] = df['Date'].map(date_to_time_idx)
df['store_id'] = df['Store'].astype(str)
df['dept_id'] = df['Dept'].astype(str)
df['Weekly_Sales'] = df['Weekly_Sales'].fillna(0)

VAL_START_DATE = '2011-12-01'
max_time_idx = df[df['Date'] < VAL_START_DATE]['time_idx'].max()
MAX_ENCODER_LENGTH = 13
MIN_PREDICT_LENGTH = 8
MAX_PREDICT_LENGTH = 8

training_dataset = TimeSeriesDataSet(
    data=df[df['time_idx'] <= max_time_idx],
    time_idx="time_idx",
    target="Weekly_Sales",
    group_ids=["store_id", "dept_id"],

    time_varying_unknown_reals=["Weekly_Sales"],

    min_encoder_length=MAX_ENCODER_LENGTH,
    max_encoder_length=MAX_ENCODER_LENGTH,
    min_prediction_length=MAX_PREDICT_LENGTH,
    max_prediction_length=MAX_PREDICT_LENGTH,

    allow_missing_timesteps=True
)

validation_dataset = TimeSeriesDataSet.from_dataset(
    training_dataset,
    data=df,
    min_prediction_idx=max_time_idx,
    max_prediction_length=MAX_PREDICT_LENGTH,
    stop_randomization=True
)

print("Datasets created successfully!")

print(f"Training samples: {len(training_dataset)}")
print(f"Validation samples: {len(validation_dataset)}")

/usr/local/lib/python3.12/dist-packages/pytorch_forecasting/data/timeseries/_timeseries.py:1861: UserWarning: Min encoder length and/or min_prediction_idx and/or min prediction length and/or lags are too large for 191 series/groups which therefore are not present in the dataset index. This means no predictions can be made for those series. First 10 removed groups: [{'__group_id__store_id': '1', '__group_id__dept_id': '77'}, {'__group_id__store_id': '1', '__group_id__dept_id': '78'}, {'__group_id__store_id': '1', '__group_id__dept_id': '99'}, {'__group_id__store_id': '10', '__group_id__dept_id': '77'}, {'__group_id__store_id': '10', '__group_id__dept_id': '78'}, {'__group_id__store_id': '11', '__group_id__dept_id': '50'}, {'__group_id__store_id': '11', '__group_id__dept_id': '77'}, {'__group_id__store_id': '11', '__group_id__dept_id': '78'}, {'__group_id__store_id': '12', '__group_id__dept_id': '51'}, {'__group_id__store_id': '12', '__group_id__dept_id': '77'}]
  warnings.warn(


Datasets created successfully!
Training samples: 215274
Validation samples: 121840


/usr/local/lib/python3.12/dist-packages/pytorch_forecasting/data/timeseries/_timeseries.py:1861: UserWarning: Min encoder length and/or min_prediction_idx and/or min prediction length and/or lags are too large for 187 series/groups which therefore are not present in the dataset index. This means no predictions can be made for those series. First 10 removed groups: [{'__group_id__store_id': '1', '__group_id__dept_id': '47'}, {'__group_id__store_id': '1', '__group_id__dept_id': '77'}, {'__group_id__store_id': '1', '__group_id__dept_id': '78'}, {'__group_id__store_id': '10', '__group_id__dept_id': '47'}, {'__group_id__store_id': '10', '__group_id__dept_id': '77'}, {'__group_id__store_id': '10', '__group_id__dept_id': '78'}, {'__group_id__store_id': '11', '__group_id__dept_id': '48'}, {'__group_id__store_id': '11', '__group_id__dept_id': '51'}, {'__group_id__store_id': '11', '__group_id__dept_id': '77'}, {'__group_id__store_id': '11', '__group_id__dept_id': '78'}]
  warnings.warn(


In [ ]:
batch_size = 32

train_dataloader = training_dataset.to_dataloader(
    batch_size=batch_size,
    num_workers=0,
    shuffle=True
)

val_dataloader = validation_dataset.to_dataloader(
    batch_size=batch_size,
    num_workers=0,
    shuffle=False
)

print("Dataloaders created!")
print(f"Train batches: {len(train_dataloader)}")
print(f"Val batches: {len(val_dataloader)}")

Dataloaders created!
Train batches: 6727
Val batches: 3807


# Training

In [ ]:
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor
from lightning.pytorch.loggers import WandbLogger
from pytorch_forecasting.models.nbeats import NBeats
import wandb
wandb.login()
wandb_logger = WandbLogger(
    project="ML_fx_N_BEATS_Walmart",
    job_type="NBEATS_Training",
    name="NBEATS_more_blocks",
    config={
        "max_encoder_length": MAX_ENCODER_LENGTH,
        "max_prediction_length": MAX_PREDICT_LENGTH,
        "widths": [32],
        "num_blocks": [3],
        "num_block_layers": [2],
        "stack_types": ["generic"],
        "learning_rate": 0.001
    }
)

model = NBeats.from_dataset(
    training_dataset,
    learning_rate=0.001,
    widths=[32],
    num_blocks=[3],
    num_block_layers=[2],
    stack_types=['generic'],
    log_interval=10
)

num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Number of trainable parameters: {num_params:,}")

early_stop_callback = EarlyStopping(monitor="val_loss", patience=3, mode="min")
lr_monitor = LearningRateMonitor(logging_interval='step')

trainer = pl.Trainer(
    max_epochs=5,
    devices=1,
    callbacks=[early_stop_callback, lr_monitor],
    logger=wandb_logger,
    enable_checkpointing=False,
    gradient_clip_val=0.1
)

trainer.fit(model, train_dataloaders=train_dataloader, val_dataloaders=val_dataloader)

print("Training Complete!")

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to th

Number of trainable parameters: 16,440


wandb: WARNING The anonymous setting has no effect and will be removed in a future version.


INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name            ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ loss            │ MASE       │      0 │ train │     0 │
│ 1 │ logging_metrics │ ModuleList │      0 │ train │     0 │
│ 2 │ net_blocks      │ ModuleList │ 16.4 K │ train │     0 │
└───┴─────────────────┴────────────┴────────┴───────┴───────┘

Trainable params: 16.4 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 16.4 K                                                                                               
Total estimated model params size (MB): 0.066                                                                      
Modules in train mode: 98                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

INFO: `Trainer.fit` stopped: `max_epochs=5` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=5` reached.


Training Complete!
